# 🛒 毛寶與競品價格監控系統 (Playwright Async 版) — 漸進式學習導覽

歡迎來到「毛寶與競品價格監控系統」實作教學！
本教學 Notebook 將原本複雜的非同步爬蟲與數據整合專案拆解為 **7 個獨立、可單獨執行的關卡**。

### 💡 學習方式與組合 .py 檔說明：
1. **每個儲存格獨立可執行**：每個 Cell 都包含獨立所需的模組 `import` 與變數，你可以隨時單獨點擊執行測試！
2. **✂️ 複製貼上標籤區塊**：每一個代碼儲存格中都清晰劃分為：
   - **`✂️ 【複製貼上區塊】`**：這是專案的模組核心（函式/設定）。未來要建立 `.py` 主程式時，**只需複製此區塊**。
   - **`🧪 【Notebook 測試區塊】`**：僅供你在 Notebook 中獨立測試該儲存格功能，不需要複製到 `.py` 檔中。

## 📌 步驟 1：環境準備與載入 JSON 設定檔

在開始爬蟲前，我們先讀取 `products_config.json`，驗證監控品類（如：手洗精、洗碗精）、毛寶自家產品、競品品牌與搜尋關鍵字。

In [ ]:
# =========================================================================
# ✂️ 【複製貼上區塊】學生未來可複製以下設定與套件載入至 .py 檔案中
# =========================================================================
import asyncio
import json
import os
import re
import urllib.parse
from datetime import datetime
from typing import Dict, List, Any
from playwright.async_api import async_playwright, BrowserContext, Page

CONFIG_FILE = "products_config.json"
REPORT_JSON = "price_report.json"
REPORT_MD = "price_report.md"
# =========================================================================

# =========================================================================
# 🧪 【Notebook 測試區塊】本儲存格獨立測試
# =========================================================================
if os.path.exists(CONFIG_FILE):
    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        config_data = json.load(f)
    print(f"✅ 成功載入設定檔！專案名稱：{config_data.get('project_name')}")
    print(f"📦 監控品類數量：{len(config_data.get('monitor_products', []))} 大類")
else:
    print(f"❌ 找不到設定檔 {CONFIG_FILE}，請確認檔案位置。")

## 📌 步驟 2：單一賣場抓取 — PChome 24h (API 方式)

PChome 提供了前端搜尋 API，我們可以利用 `context.request.get()` 直接發送 HTTP 請求並取得 JSON 數據，不需要繪製 UI 畫面，速度極快！

**學習目標**：定義 `fetch_pchome` 函式，並單獨測試它。

In [ ]:
# =========================================================================
# ✂️ 【複製貼上區塊】學生未來可複製以下函式至 .py 檔案中
# =========================================================================
import urllib.parse
from typing import Dict, Any
from playwright.async_api import BrowserContext

async def fetch_pchome(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    """從 PChome 24h 購物抓取第一筆商品資訊 (API 方式)"""
    result = {
        "platform": "PChome 24h",
        "title": "未找到相關商品",
        "price": 0,
        "url": "",
        "status": "無結果"
    }

    encoded_kw = urllib.parse.quote(keyword)
    api_url = f"https://ecshweb.pchome.com.tw/search/v3.3/all/results?q={encoded_kw}&page=1"

    try:
        response = await context.request.get(api_url, timeout=10000)
        if response.status == 200:
            data = await response.json()
            prods = data.get("prods", [])
            if prods:
                item = prods[0]
                result["title"] = item.get("name", "未知的商品標題")
                result["price"] = int(item.get("price", 0))
                result["url"] = f"https://24h.pchome.com.tw/prod/{item.get('Id', '')}"
                result["status"] = "成功"
                return result
    except Exception as e:
        print(f"PChome 抓取失敗: {e}")

    return result
# =========================================================================

# =========================================================================
# 🧪 【Notebook 測試區塊】本儲存格獨立測試
# =========================================================================
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context()
    test_res = await fetch_pchome(context, "毛寶 貼身衣物手洗精")
    print("🧪 PChome 單獨測試結果：", test_res)
    await browser.close()

## 📌 步驟 3：單一賣場抓取 — momo 購物網 (Playwright DOM 解析)

momo 購物網需要實體瀏覽器渲染網頁，我們透過 `context.new_page()` 開啟頁面，並使用 `locator` 定位商品標題與價格。

**學習目標**：定義 `fetch_momo` 函式並單獨測試。

In [ ]:
# =========================================================================
# ✂️ 【複製貼上區塊】學生未來可複製以下函式至 .py 檔案中
# =========================================================================
import re
import urllib.parse
from typing import Dict, Any
from playwright.async_api import BrowserContext, Page

async def fetch_momo(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    """從 momo購物網 抓取第一筆商品資訊 (DOM 解析)"""
    result = {
        "platform": "momo購物網",
        "title": "未找到相關商品",
        "price": 0,
        "url": "",
        "status": "無結果"
    }

    encoded_kw = urllib.parse.quote(keyword)
    url = f"https://www.momoshop.com.tw/search/searchShop.jsp?keyword={encoded_kw}"
    page: Page = await context.new_page()

    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=15000)
        await page.wait_for_timeout(1000)
        cards = page.locator("div.listArea ul li, .prdListArea ul li")
        
        if await cards.count() > 0:
            card = cards.first
            title_loc = card.locator(".prdName, h3, .goodsName")
            price_loc = card.locator(".price, .money, .prdPrice")
            link_loc = card.locator("a.goods-img-url, a.prdName, a").first

            title = await title_loc.first.inner_text() if await title_loc.count() > 0 else ""
            price_text = await price_loc.first.inner_text() if await price_loc.count() > 0 else ""
            href = await link_loc.get_attribute("href") if await link_loc.count() > 0 else ""

            digits = re.sub(r"[^\d]", "", price_text)
            price = int(digits) if digits else 0

            if href and not href.startswith("http"):
                href = f"https://www.momoshop.com.tw{href}"

            if title:
                result["title"] = title.strip()
                result["price"] = price
                result["url"] = href
                result["status"] = "成功"
    except Exception as e:
        print(f"momo 抓取失敗: {e}")
    finally:
        await page.close()

    return result
# =========================================================================

# =========================================================================
# 🧪 【Notebook 測試區塊】本儲存格獨立測試
# =========================================================================
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    test_res = await fetch_momo(context, "毛寶 貼身衣物手洗精")
    print("🧪 momo 單獨測試結果：", test_res)
    await browser.close()

## 📌 步驟 4：單一賣場抓取 — Yahoo 購物中心 (Playwright DOM 解析)

Yahoo 購物中心同樣使用 Playwright 抓取 DOM。這裡示範了如何解析卡片內部文字行，分離標題與包含 `$` 的價格字串。

**學習目標**：定義 `fetch_yahoo` 函式並單獨測試。

In [ ]:
# =========================================================================
# ✂️ 【複製貼上區塊】學生未來可複製以下函式至 .py 檔案中
# =========================================================================
import re
import urllib.parse
from typing import Dict, Any
from playwright.async_api import BrowserContext, Page

async def fetch_yahoo(context: BrowserContext, keyword: str) -> Dict[str, Any]:
    """從 Yahoo購物中心 抓取第一筆商品資訊 (DOM 解析)"""
    result = {
        "platform": "Yahoo購物中心",
        "title": "未找到相關商品",
        "price": 0,
        "url": "",
        "status": "無結果"
    }

    encoded_kw = urllib.parse.quote(keyword)
    url = f"https://tw.buy.yahoo.com/search/product?p={encoded_kw}"
    page: Page = await context.new_page()

    try:
        await page.goto(url, wait_until="domcontentloaded", timeout=15000)
        await page.wait_for_timeout(1200)
        cards = page.locator("a[href*='/gdsale/']")

        if await cards.count() > 0:
            card = cards.first
            href = await card.get_attribute("href")
            txt = await card.inner_text()
            lines = [l.strip() for l in txt.split("\n") if l.strip()]

            title = ""
            price = 0
            for l in lines:
                if l.startswith("$"):
                    digits = re.sub(r"[^\d]", "", l)
                    if digits and price == 0:
                        price = int(digits)
                elif l not in ["比較", "找相似", "活動", "券", "限時下殺", "折扣"] and not title:
                    title = l

            if title:
                result["title"] = title
                result["price"] = price
                result["url"] = href or ""
                result["status"] = "成功"
    except Exception as e:
        print(f"Yahoo 抓取失敗: {e}")
    finally:
        await page.close()

    return result
# =========================================================================

# =========================================================================
# 🧪 【Notebook 測試區塊】本儲存格獨立測試
# =========================================================================
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    test_res = await fetch_yahoo(context, "毛寶 貼身衣物手洗精")
    print("🧪 Yahoo 單獨測試結果：", test_res)
    await browser.close()

## 📌 步驟 5：跨賣場非同步併發 (asyncio.gather)

如果依序查詢 PChome -> momo -> Yahoo，總共需要花費 3 倍的時間。
透過 `asyncio.gather(*tasks)`，我們可以同時向三個電商平台發起搜尋 request，時間縮短為最慢平台的響應時間！

**學習目標**：定義 `fetch_item_across_platforms` 函式，整合 PChome、momo、Yahoo 三大賣場的平行併發。

In [ ]:
# =========================================================================
# ✂️ 【複製貼上區塊】學生未來可複製以下函式至 .py 檔案中
# =========================================================================
import asyncio
from typing import Dict, Any
from playwright.async_api import BrowserContext

async def fetch_item_across_platforms(context: BrowserContext, brand: str, name: str, keyword: str) -> Dict[str, Any]:
    """在各大賣場 (PChome, momo, Yahoo) 平行併發查詢該商品資訊"""
    stores_tasks = [
        fetch_pchome(context, keyword),
        fetch_momo(context, keyword),
        fetch_yahoo(context, keyword)
    ]
    # 同時執行三個賣場的爬取任務
    store_results = await asyncio.gather(*stores_tasks)

    return {
        "brand": brand,
        "name": name,
        "keyword": keyword,
        "stores": store_results
    }
# =========================================================================

# =========================================================================
# 🧪 【Notebook 測試區塊】本儲存格獨立測試
# =========================================================================
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    print("🚀 開始同時查詢 3 大賣場...")
    test_res = await fetch_item_across_platforms(context, "毛寶", "毛寶 貼身衣物手洗精 1000g", "毛寶 貼身衣物手洗精")
    print("🧪 跨賣場併發結果：")
    for s in test_res["stores"]:
        print(f" - [{s['platform']}] {s['title']} | 售價: ${s['price']}")
    await browser.close()

## 📌 步驟 6：品類與競品併發監控 (Category Level Monitoring)

在單一品類中（例如「貼身衣物手洗精」），既有毛寶自家產品，也有競品（如白蘭、妙管家）。
我們使用 `monitor_category_async` 來同時對「毛寶」與「所有競品」進行跨平台的併發查詢。

**學習目標**：理解多層次 `asyncio.gather` 的架構。

In [ ]:
# =========================================================================
# ✂️ 【複製貼上區塊】學生未來可複製以下函式至 .py 檔案中
# =========================================================================
import asyncio
from typing import Dict, Any
from playwright.async_api import BrowserContext

async def monitor_category_async(context: BrowserContext, category_item: Dict[str, Any]) -> Dict[str, Any]:
    """非同步處理單一品類（包含毛寶與競品，平行抓取各大賣場）"""
    category_name = category_item["category"]
    maobao_cfg = category_item["maobao_product"]
    competitors_cfg = category_item["competitors"]

    print(f"🚀 開始平行併發查詢品類：【{category_name}】跨賣場數據...")

    # 1. 建立毛寶產品的抓取任務
    tasks = [
        fetch_item_across_platforms(context, "毛寶", maobao_cfg["name"], maobao_cfg["keyword"])
    ]
    # 2. 建立所有競品的抓取任務
    for comp in competitors_cfg:
        tasks.append(fetch_item_across_platforms(context, comp["brand"], comp["name"], comp["keyword"]))

    # 3. 同時發起所有產品的跨賣場搜尋！
    results = await asyncio.gather(*tasks)

    return {
        "category": category_name,
        "maobao_product": results[0],
        "competitors": results[1:]
    }
# =========================================================================

# =========================================================================
# 🧪 【Notebook 測試區塊】本儲存格獨立測試
# =========================================================================
import json
import os
from playwright.async_api import async_playwright

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context(user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36")
    
    # 獨立載入設定檔測試
    if os.path.exists("products_config.json"):
        with open("products_config.json", "r", encoding="utf-8") as f:
            test_cfg = json.load(f)
            first_category = test_cfg["monitor_products"][0]
            cat_res = await monitor_category_async(context, first_category)
            print(f"✅ 完成品類【{cat_res['category']}】的跨賣場與競品監控！")
    await browser.close()

## 📌 步驟 7：全系統整合與組合 Python 檔 (.py) 導覽

🎉 **恭喜完成所有單元開發與測試！**

### 🧩 學生如何組裝成完整的 Python (.py) 專案檔？
學生未來只需將上述步驟的 **`✂️ 【複製貼上區塊】`** 按順序複製貼到一個新建的 `main.py` 檔案中：
1. **最頂層**：複製步驟 1 的所有 `import` 與常數定義（`CONFIG_FILE`, `REPORT_JSON`, `REPORT_MD`）。
2. **賣場抓取模組**：複製步驟 2 (`fetch_pchome`)、步驟 3 (`fetch_momo`)、步驟 4 (`fetch_yahoo`) 的函式。
3. **非同步併發模組**：複製步驟 5 (`fetch_item_across_platforms`) 與步驟 6 (`monitor_category_async`) 的函式。
4. **主程式與報表產出**：複製本步驟（步驟 7）的 `main()` 函式與 `if __name__ == "__main__": asyncio.run(main())`！

在 Notebook 中，你可以直接執行下方 Cell 來跑完全系統的自動化監控與報表輸出：

In [ ]:
# =========================================================================
# ✂️ 【複製貼上區塊】學生未來可複製以下 main 函式與報表輸出至 .py 檔案中
# =========================================================================
async def main():
    print("=" * 80)
    print("毛寶企業 (Maobao) 多賣場產品與競品價格每日監控系統 [Playwright Async 版]")
    print("=" * 80)

    if not os.path.exists(CONFIG_FILE):
        print(f"❌ 找不到設定檔：{CONFIG_FILE}")
        return

    with open(CONFIG_FILE, "r", encoding="utf-8") as f:
        config_data = json.load(f)

    categories = config_data.get("monitor_products", [])
    platforms = config_data.get("platforms", [])
    platform_names = [p["name"] for p in platforms]

    print(f"📦 載入設定完成！監控 {len(categories)} 大品類，跨賣場：{', '.join(platform_names)}...\n")

    start_time = datetime.now()

    # 啟動 Playwright 瀏覽器環境
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context: BrowserContext = await browser.new_context(
            viewport={"width": 1280, "height": 720},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36"
        )

        cat_tasks = [monitor_category_async(context, cat) for cat in categories]
        all_results = await asyncio.gather(*cat_tasks)

        await context.close()
        await browser.close()

    elapsed = (datetime.now() - start_time).total_seconds()
    print("\n" + "=" * 80)
    print(f"✓ 所有品類跨賣場抓取完成！總耗時僅：{elapsed:.2f} 秒 (非同步併發加速)")
    print("=" * 80)

    # 輸出終端機表格
    print("\n📊 【毛寶 vs 競品 多賣場即時價格監控報表】")
    print("註：因各商品包裝規格與單位不同，本報表僅呈現原始監控售價，不進行價差比較與優劣分析")
    print("-" * 90)
    print(f"{'品類':<12} {'品牌':<8} {'賣場':<12} {'搜尋商品標題':<32} {'售價'}")
    print("-" * 90)

    for cat_data in all_results:
        cat_name = cat_data["category"]
        all_prods = [cat_data["maobao_product"]] + cat_data["competitors"]

        for prod_idx, prod in enumerate(all_prods):
            brand_label = f"[{prod['brand']}]"
            is_first_store = True

            for store in prod["stores"]:
                title_short = (store["title"][:30] + "..") if len(store["title"]) > 30 else store["title"]
                price_str = f"${store['price']}" if store["price"] > 0 else "未找到"
                
                cat_disp = cat_name if (prod_idx == 0 and is_first_store) else ""
                brand_disp = brand_label if is_first_store else ""

                print(f"{cat_disp:<12} {brand_disp:<8} {store['platform']:<12} {title_short:<32} {price_str}")
                is_first_store = False
        print("-" * 90)

    # 匯出 JSON 詳細數據
    report_json_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "elapsed_seconds": round(elapsed, 2),
        "platforms": platform_names,
        "note": "單位與包裝規格不同，無輸出價差比較與優劣分析",
        "data": all_results
    }
    with open(REPORT_JSON, "w", encoding="utf-8") as f:
        json.dump(report_json_data, f, ensure_ascii=False, indent=2)
    print(f"\n✓ 已匯出 JSON 詳細數據：{REPORT_JSON}")

    # 匯出 Markdown 報表
    md_content = f"# 毛寶企業 產品與競品多賣場價格監控日報\n\n"
    md_content += f"- **監控時間**：{report_json_data['timestamp']}\n"
    md_content += f"- **總耗時**：{elapsed:.2f} 秒 (Playwright Async 多賣場平行併發)\n"
    md_content += f"- **監控賣場**：{', '.join(platform_names)}\n"
    md_content += f"- **說明**：*因各品牌商品與包裝規格單位不一，本報告僅呈現各平台即時監控價格與標題，不進行價差比較。*\n\n"
    md_content += f"## 📊 跨賣場價格一覽表\n\n"
    md_content += f"| 品類 | 品牌 | 賣場平台 | 搜尋商品標題 | 售價 (TWD) | 商品連結 |\n"
    md_content += f"| :--- | :--- | :--- | :--- | :--- | :--- |\n"

    for cat_data in all_results:
        cat_name = cat_data["category"]
        all_prods = [cat_data["maobao_product"]] + cat_data["competitors"]

        for prod in all_prods:
            brand_label = f"**{prod['brand']}**" if prod['brand'] == '毛寶' else prod['brand']
            for store in prod["stores"]:
                price_disp = f"**${store['price']}**" if store["price"] > 0 else "N/A"
                url_disp = f"[商品連結]({store['url']})" if store['url'] else "N/A"
                md_content += f"| {cat_name} | {brand_label} | {store['platform']} | {store['title']} | {price_disp} | {url_disp} |\n"

    with open(REPORT_MD, "w", encoding="utf-8") as f:
        f.write(md_content)
    print(f"✓ 已匯出 Markdown 分析報告：{REPORT_MD}")
# =========================================================================

# 🚀 在 Notebook 中直接執行主程式
if __name__ == "__main__":
    await main()